## Importing libraries 

In [ ]:
import pandas as pd
import numpy as np

import warnings 
warnings.filterwarnings("ignore")

In [ ]:
data=pd.read_csv('combined_sample.csv')
df=data.copy()
df.head()

## Data Understanding

In [ ]:
# Basic data info
df.info()

In [ ]:
# Checking the total null values count in the each columns 
df.isna().sum()

In [ ]:
# Sort the data based on year and month
df=df.sort_values(by=['YEAR','MONTH'],ascending=True)

In [ ]:
# if flight had a column where it was delayed or diverted and the air time was null removing those rows
mask = (
    df['AIR_TIME'].isna() &
    ((df['CANCELLED'] == 1) | (df['DIVERTED'] == 1))
)

df = df[~mask]

In [ ]:
# Dropping null values
df = df.dropna(
    subset=['TAXI_IN', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME']
)

In [ ]:
# Filling the null values
delay_cols = [
    'CARRIER_DELAY',
    'WEATHER_DELAY',
    'NAS_DELAY',
    'SECURITY_DELAY',
    'LATE_AIRCRAFT_DELAY'
]

# rows where any delay exists
df[delay_cols]= df[delay_cols].fillna(0)


In [ ]:
# Checking the total null values count in the each columns 
df.isna().sum()

In [ ]:
# Check for duplicate rows
df.duplicated().sum()

In [ ]:
# Dropping the duplicated rows
df.drop_duplicates(inplace=True)

## Feature Engineering 

In [ ]:
df['operational_risk'] = (
    (df['CARRIER_DELAY'] > 0) |
    (df['LATE_AIRCRAFT_DELAY'] > 0) |
    (df['NAS_DELAY'] > 0) |
    (df['SECURITY_DELAY'] > 0)|
    (df['WEATHER_DELAY'] > 0)
).astype(int)

In [ ]:
# Checking for the class balance of target variable 
df['operational_risk'].value_counts(normalize=True)

In [ ]:
# Checking the shape of the data year wise
train_df = df[df['YEAR'] == 2024]
test_df  = df[df['YEAR'] == 2025]
print(train_df.shape, test_df.shape)

In [ ]:
# Drawing a stratified sample from the 2024 dataset for model training

target_n = 200001

train_sample = (
    train_df
    .groupby('operational_risk', group_keys=False)
    .apply(lambda x: x.sample(
        n=int(len(x) / len(train_df) * target_n),
        random_state=42
    ))
    .reset_index(drop=True)
)


In [ ]:
# Drawing a stratified sample from the 2025 dataset for model testing
target_n = 100001

test_sample = (
    test_df
    .groupby('operational_risk', group_keys=False)
    .apply(lambda x: x.sample(
        n=int(len(x) / len(test_df) * target_n),
        random_state=42
    ))
    .reset_index(drop=True)
)


In [ ]:
print(train_sample.shape)
print(test_sample.shape)

In [ ]:
# Extracting the train and test samples into csv
train_sample.to_csv("train_sample_2024.csv", index=False)
test_sample.to_csv("test_sample_2025.csv", index=False)